<a href="https://colab.research.google.com/github/Romanzilla/ML/blob/main/%D0%9B%D0%A0_9_%D0%9D%D0%B0%D0%B4%D1%96%D0%B9%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Надійко Роман Олександрович
**Група:** ФІТ 3-10
**Дата:** 17.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [1]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.2.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [2]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [3]:
# TODO 2.1 — Дослідіть набір даних:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {n_users}")
print(f"Кількість унікальних фільмів: {n_items}")
print(f"Мінімальна оцінка: {min_rating}")
print(f"Максимальна оцінка: {max_rating}")
print(f"Середня оцінка: {mean_rating:.4f}")
print("\nРозподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())
print("\nПропущені значення за стовпцями:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Пропущені значення за стовпцями:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних `MovieLens 100K` містить **100 000 оцінок** від **943 унікальних користувачів** для **1 682 фільмів**, пропущені значення повністю відсутні. Оцінки задані за цілочисельною шкалою від `1.0` до `5.0`, при цьому середній рейтинг становить `3.5299`. Розподіл зміщений у бік позитивних оцінок: найчастіше користувачі ставлять `4.0` (34 174 рази) та `3.0` (27 145 разів), тоді як мінімальна оцінка `1.0` зустрічається найрідше (6 110 разів).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [4]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"Розмір train_df: {train_df.shape}")
print(f"Розмір test_df:  {test_df.shape}")

Розмір train_df: (80000, 4)
Розмір test_df:  (20000, 4)


In [5]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print(f"Кількість записів у train_data: {len(train_df)}")
print(f"Кількість записів у testset:    {len(testset)}")
print("Приклад елемента testset:", testset[0])

Кількість записів у train_data: 80000
Кількість записів у testset:    20000
Приклад елемента testset: ('877', '381', 4.0)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [6]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

* **Для моделі `SVD` (матрична факторизація):**
  * `n_factors` (`[50, 100]`): визначає кількість прихованих (латентних) факторів, що описують смаки користувачів і характеристики фільмів; порівняння 50 та 100 факторів дозволяє знайти баланс між виразністю моделі та ризиком перенавчання.
  * `lr_all` (`[0.005, 0.010]`): крок стохастичного градієнтного спуску; помірні значення забезпечують стабільну збіжність без розходження оптимізації.
  * `reg_all` (`[0.02, 0.05]`): коефіцієнт $L_2$-регуляризації, який стримує зростання ваг і запобігає перенавчанню на розрідженій матриці оцінок.
* **Для моделі `KNNBasic` (колаборативна фільтрація на основі сусідів):**
  * `k` (`[20, 40]`): максимальна кількість найближчих сусідів, за якими обчислюється зважений прогноз рейтингу.
  * `min_k` (`[1, 5]`): мінімальний поріг сусідів; значення `5` захищає від ненадійних прогнозів, зроблених за одним випадковим збігом.
  * `sim_options` (`'name': ['cosine', 'pearson']`, `'user_based': [False, True]`): дозволяє порівняти косинусну подібність із кореляцією Пірсона (яка враховує індивідуальну суворість оцінок користувачів), а також зіставити пошук схожих фільмів (*item-based*, `False`) і схожих користувачів (*user-based*, `True`).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [7]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)
print("Пошук гіперпараметрів для SVD завершено.")

Пошук гіперпараметрів для SVD завершено.


In [8]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)
print("Пошук гіперпараметрів для KNNBasic завершено.")

Пошук гіперпараметрів для KNNBasic завершено.


In [9]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
best_idx_svd = grid_search_svd.best_index['rmse']
svd_cv_rmse = grid_search_svd.best_score['rmse']
svd_cv_mae = grid_search_svd.cv_results['mean_test_mae'][best_idx_svd]
best_svd_params = grid_search_svd.best_params['rmse']

best_idx_knn = grid_search_knn.best_index['rmse']
knn_cv_rmse = grid_search_knn.best_score['rmse']
knn_cv_mae = grid_search_knn.cv_results['mean_test_mae'][best_idx_knn]
best_knn_params = grid_search_knn.best_params['rmse']

print("=== SVD (GridSearchCV) ===")
print(f"Найкращий CV RMSE: {svd_cv_rmse:.4f}")
print(f"CV MAE (для моделі з найкращим RMSE): {svd_cv_mae:.4f} | Найкращий CV MAE: {grid_search_svd.best_score['mae']:.4f}")
print(f"Найкращі параметри за RMSE: {best_svd_params}")

print("\n=== KNNBasic (GridSearchCV) ===")
print(f"Найкращий CV RMSE: {knn_cv_rmse:.4f}")
print(f"CV MAE (для моделі з найкращим RMSE): {knn_cv_mae:.4f} | Найкращий CV MAE: {grid_search_knn.best_score['mae']:.4f}")
print(f"Найкращі параметри за RMSE: {best_knn_params}")

=== SVD (GridSearchCV) ===
Найкращий CV RMSE: 0.9433
CV MAE (для моделі з найкращим RMSE): 0.7434 | Найкращий CV MAE: 0.7434
Найкращі параметри за RMSE: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}

=== KNNBasic (GridSearchCV) ===
Найкращий CV RMSE: 1.0272
CV MAE (для моделі з найкращим RMSE): 0.8150 | Найкращий CV MAE: 0.8150
Найкращі параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | <SVD_CV_RMSE> | <SVD_CV_MAE> | <BEST_SVD_PARAMS> |
| KNNBasic | <KNN_CV_RMSE> | <KNN_CV_MAE> | <BEST_KNN_PARAMS> |

На перехресній перевірці модель **SVD** показала нижчу похибку за обома метриками (RMSE та MAE), ніж **KNNBasic**. Матрична факторизація краще справляється з розрідженістю матриці оцінок `MovieLens 100K`, виділяючи латентні вподобання користувачів, тоді як класичний `KNNBasic` чутливіший до малого перетину спільно оцінених фільмів.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [10]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 — Створіть trainset із train_data.
trainset = train_data.build_full_trainset()

# TODO 6.3 — Навчіть best_svd і best_knn.
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("Остаточні моделі best_svd та best_knn навчено на повній навчальній вибірці.")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Остаточні моделі best_svd та best_knn навчено на повній навчальній вибірці.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [11]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f"SVD      -> Test RMSE: {svd_rmse:.4f}, Test MAE: {svd_mae:.4f}")
print(f"KNNBasic -> Test RMSE: {knn_rmse:.4f}, Test MAE: {knn_mae:.4f}")

SVD      -> Test RMSE: 0.9194, Test MAE: 0.7247
KNNBasic -> Test RMSE: 1.0120, Test MAE: 0.7984


In [12]:
# TODO 7.3 — Сформуйте DataFrame results із колонками: Model, RMSE, MAE.
results = (
    pd.DataFrame({
        'Model': ['SVD', 'KNNBasic'],
        'RMSE': [svd_rmse, knn_rmse],
        'MAE': [svd_mae, knn_mae]
    })
    .sort_values(by='RMSE', ascending=True)
    .reset_index(drop=True)
)

display(results)

,Model,RMSE,MAE
0,SVD,0.919367,0.724672
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | <SVD_TEST_RMSE> | <SVD_TEST_MAE> |
| KNNBasic | <KNN_TEST_RMSE> | <KNN_TEST_MAE> |

**Порівняння з CV:** Значення RMSE та MAE на незалежній тестовій вибірці узгоджуються з результатами 3-блочної кросвалідації та навіть є дещо кращими (похибка трохи менша), оскільки остаточні моделі навчалися на 100% навчальної вибірки замість двох третин під час CV. Невелика різниця між CV та тестом підтверджує відсутність перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [13]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
if svd_cv_rmse <= knn_cv_rmse:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Обрана модель для рекомендацій (за кращим CV RMSE): {best_model_name}")

# TODO 8.2 — Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)

    # TODO 8.3 — Знайдіть неоцінені ним фільми.
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    rated_raw_iids = {trainset.to_raw_iid(iid) for iid in rated_inner_iids}

    all_raw_iids = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]
    unrated_raw_iids = [iid for iid in all_raw_iids if iid not in rated_raw_iids]

    print(f"Користувач {user_id} вже оцінив у навчальних даних {len(rated_raw_iids)} фільмів.")
    print(f"Кількість неоцінених фільмів-кандидатів: {len(unrated_raw_iids)}")

    # TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
    predictions_for_user = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    top_10_df = (
        pd.DataFrame(predictions_for_user, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    top_10_df.index = top_10_df.index + 1
    display(top_10_df)

except ValueError:
    print(f"Попередження: користувача з user_id='{user_id}' не знайдено у trainset (проблема холодного старту).")

Обрана модель для рекомендацій (за кращим CV RMSE): SVD
Користувач 196 вже оцінив у навчальних даних 29 фільмів.
Кількість неоцінених фільмів-кандидатів: 1624


,item_id,predicted_rating
1,318,4.578034
2,427,4.510453
3,603,4.455609
4,1169,4.441489
5,661,4.414566
6,479,4.390852
7,513,4.356216
8,357,4.324538
9,64,4.322364
10,313,4.300572


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

**Висновок:** У ході лабораторної роботи на наборі даних `MovieLens 100K` (100 000 оцінок, 943 користувачі, 1 682 фільми) було побудовано та порівняно дві рекомендаційні моделі — `SVD` і `KNNBasic`. За результатами 3-блочної кросвалідації найкращою моделлю обрано **SVD**, оскільки вона продемонструвала нижчу помилку (`CV RMSE = <SVD_CV_RMSE>`, `CV MAE = <SVD_CV_MAE>`) порівняно з `KNNBasic` (`CV RMSE = <KNN_CV_RMSE>`, `CV MAE = <KNN_CV_MAE>`). Оптимальними гіперпараметрами стали `<BEST_SVD_PARAMS>` для SVD та `<BEST_KNN_PARAMS>` для KNNBasic. На незалежній тестовій вибірці SVD також підтвердила перевагу з показниками `Test RMSE = <SVD_TEST_RMSE>` та `Test MAE = <SVD_TEST_MAE>` проти `Test RMSE = <KNN_TEST_RMSE>` і `Test MAE = <KNN_TEST_MAE>` у KNNBasic. Невелике покращення метрик на тесті порівняно з CV пояснюється навчанням фінальних моделей на повній навчальній вибірці (80 000 оцінок) та свідчить про стабільну узагальнювальну здатність без перенавчання. Основними обмеженнями побудованої системи колаборативної фільтрації є проблема «холодного старту» (неможливість сформувати персоналізовані рекомендації для нових користувачів або нових фільмів без історії оцінок) та відсутність контентних ознак фільмів (жанрів, року випуску, акторів).

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [14]:
# власна реалізація Biased SVD (Matrix Factorization через SGD) з нуля
class CustomBiasedSVD:
    def __init__(self, n_factors=50, lr=0.007, reg=0.05, n_epochs=20, min_rating=1.0, max_rating=5.0, random_state=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_rating = min_rating
        self.max_rating = max_rating
        self.random_state = random_state

    def fit(self, df_train):
        rng = np.random.default_rng(self.random_state)

        self.user_to_idx = {u: idx for idx, u in enumerate(df_train['user_id'].unique())}
        self.item_to_idx = {i: idx for idx, i in enumerate(df_train['item_id'].unique())}
        self.idx_to_item = {idx: i for i, idx in self.item_to_idx.items()}

        n_users = len(self.user_to_idx)
        n_items = len(self.item_to_idx)

        self.mu = df_train['rating'].mean()
        self.b_u = np.zeros(n_users, dtype=np.float64)
        self.b_i = np.zeros(n_items, dtype=np.float64)
        self.P = rng.normal(0.0, 0.1, size=(n_users, self.n_factors))
        self.Q = rng.normal(0.0, 0.1, size=(n_items, self.n_factors))

        u_indices = df_train['user_id'].map(self.user_to_idx).to_numpy()
        i_indices = df_train['item_id'].map(self.item_to_idx).to_numpy()
        ratings = df_train['rating'].to_numpy(dtype=np.float64)

        n_samples = len(ratings)
        for _ in range(self.n_epochs):
            perm = rng.permutation(n_samples)
            for idx in perm:
                u = u_indices[idx]
                i = i_indices[idx]
                r = ratings[idx]

                pred = self.mu + self.b_u[u] + self.b_i[i] + np.dot(self.P[u], self.Q[i])
                err = r - pred

                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])

                p_u_old = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_u_old - self.reg * self.Q[i])

        return self

    def predict_one(self, raw_user_id, raw_item_id):
        u = self.user_to_idx.get(raw_user_id)
        i = self.item_to_idx.get(raw_item_id)

        est = self.mu
        if u is not None:
            est += self.b_u[u]
        if i is not None:
            est += self.b_i[i]
        if u is not None and i is not None:
            est += float(np.dot(self.P[u], self.Q[i]))

        return float(np.clip(est, self.min_rating, self.max_rating))

    def evaluate(self, df_test):
        preds = np.array([
            self.predict_one(u, i)
            for u, i in zip(df_test['user_id'], df_test['item_id'])
        ])
        actuals = df_test['rating'].to_numpy(dtype=np.float64)
        rmse = np.sqrt(np.mean((actuals - preds) ** 2))
        mae = np.mean(np.abs(actuals - preds))
        return rmse, mae

    def recommend_top_n(self, raw_user_id, df_train, n=10):
        if raw_user_id not in self.user_to_idx:
            return pd.DataFrame(columns=['item_id', 'predicted_rating'])

        rated_items = set(df_train.loc[df_train['user_id'] == raw_user_id, 'item_id'])
        candidates = [item for item in self.item_to_idx if item not in rated_items]

        scored = [(item, self.predict_one(raw_user_id, item)) for item in candidates]
        top_df = (
            pd.DataFrame(scored, columns=['item_id', 'predicted_rating'])
            .sort_values(by='predicted_rating', ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        top_df.index = top_df.index + 1
        return top_df


# Навчання та перевірка власного алгоритму на тих самих train_df та test_df
custom_svd = CustomBiasedSVD(n_factors=50, lr=0.007, reg=0.05, n_epochs=20, random_state=42)
custom_svd.fit(train_df)

custom_rmse, custom_mae = custom_svd.evaluate(test_df)
print(f"CustomBiasedSVD (з нуля) -> Test RMSE: {custom_rmse:.4f}, Test MAE: {custom_mae:.4f}")
print("\nТоп-10 рекомендацій власного алгоритму для користувача '196':")
display(custom_svd.recommend_top_n('196', train_df, n=10))

CustomBiasedSVD (з нуля) -> Test RMSE: 0.9228, Test MAE: 0.7280

Топ-10 рекомендацій власного алгоритму для користувача '196':


,item_id,predicted_rating
1,318,4.421962
2,114,4.403241
3,64,4.402844
4,483,4.366464
5,603,4.344551
6,1194,4.340996
7,513,4.337784
8,98,4.331476
9,480,4.309100
10,174,4.279745
